In [3]:
def split_markdown_by_titles(markdown_text: str, max_chars: int) -> list[str]:
    """
    Split markdown text into groups based on title hierarchy and character limits.
    
    Args:
        markdown_text: Input markdown text
        max_chars: Maximum characters allowed per group
    
    Returns:
        List of markdown text groups
    """
    def count_heading_level(line: str) -> int:
        """Count the number of '#' at start of line to determine heading level"""
        if not line.startswith('#'):
            return 0
        return len(line) - len(line.lstrip('#'))

    def split_content(text: str, level: int = 1) -> list[list[str]]:
        """Recursively split content by heading levels"""
        if not text:
            return []

        lines = text.split('\n')
        sections = []
        current_section = []
        
        for line in lines:
            heading_level = count_heading_level(line)
            
            if heading_level == level:
                if current_section:
                    sections.append(current_section)
                current_section = [line]
            else:
                current_section.append(line)
                
        if current_section:
            sections.append(current_section)
            
        # If we're at the deepest level, return sections
        if all(count_heading_level(section[0]) != level + 1 for section in sections):
            return sections
            
        # Otherwise, recursively split each section
        result = []
        for section in sections:
            subsections = split_content('\n'.join(section), level + 1)
            result.extend(subsections)
            
        return result

    # Get initial splits by titles
    sections = split_content(markdown_text)
    
    # Group lines within each section based on max_chars
    final_groups = []
    for section in sections:
        current_group = []
        current_length = 0
        
        for line in section:
            line_length = len(line)
            
            if current_length + line_length > max_chars and current_group:
                final_groups.append('\n'.join(current_group))
                current_group = []
                current_length = 0
                
            current_group.append(line)
            current_length += line_length
            
        if current_group:
            final_groups.append('\n'.join(current_group))
            
    return final_groups

In [9]:
markdown_text = """
# Title 1
Some content
## Subtitle 1.1
More content
More contentMore contentMore content
More contentMore content
More contentMore contentMore content
More content
More content
More content
More content
## Subtitle 1.2
Even more content
More content
More content
More content
More content

# Title 2
## Subtitle 2.1
More content
More content
More content
Final content
312321
31231
321312
3213213131
321321312312312
## Subtitle 2.2
More content
More content
More content
More content
## Subtitle 2.3
More content
More content
More content
"""

groups = split_markdown_by_titles(markdown_text, max_chars=100)
print("\n-------\n".join(groups))


-------
# Title 1
Some content
## Subtitle 1.1
More content
More contentMore contentMore content
-------
More contentMore content
More contentMore contentMore content
More content
More content
More content
-------
More content
## Subtitle 1.2
Even more content
More content
More content
More content
More content

-------
# Title 2
## Subtitle 2.1
More content
More content
More content
Final content
312321
31231
321312
3213213131
-------
321321312312312
## Subtitle 2.2
More content
More content
More content
More content
## Subtitle 2.3
-------
More content
More content
More content

